<a href="https://colab.research.google.com/github/swarnkarnitin/TrafficMonitoring/blob/main/yolo_training_vehicle_traffic_signal_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
import ultralytics.data.build as build
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

In [7]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        """
        Initialize the WeightedDataset.

        Args:
            class_weights (list or numpy array): A list or array of weights corresponding to each class.
        """

        super(YOLOWeightedDataset, self).__init__(*args, **kwargs)

        self.train_mode = "train" in self.prefix

        # You can also specify weights manually instead
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts

        # Aggregation function
        self.agg_func = np.mean

        self.class_weights = np.array(class_weights)
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()

    def count_instances(self):
        """
        Count the number of instances per class

        Returns:
            dict: A dict containing the counts for each class.
        """
        self.counts = [0 for i in range(len(self.data["names"]))]
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            for id in cls:
                self.counts[id] += 1

        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        """
        Calculate the aggregated weight for each label based on class weights.

        Returns:
            list: A list of aggregated weights corresponding to each label.
        """
        weights = []
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)

            # Give a default weight to background class
            if cls.size == 0:
              weights.append(1)
              continue

            # Take mean of weights
            # You can change this weight aggregation function to aggregate weights differently
            weight = self.agg_func(self.class_weights[cls])
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        """
        Calculate and store the sampling probabilities based on the weights.

        Returns:
            list: A list of sampling probabilities corresponding to each label.
        """
        total_weight = sum(self.weights)
        probabilities = [w / total_weight for w in self.weights]
        return probabilities

    def __getitem__(self, index):
        """
        Return transformed label information based on the sampled index.
        """
        # Don't use for validation
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            index = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(index))

In [8]:
build.YOLODataset = YOLOWeightedDataset

In [ ]:
!cp -r /content/drive/MyDrive/Traffic_Videos/resultant_dataset ./

In [12]:


# Load a YOLOv8n model
# model = YOLO("/content/drive/MyDrive/Traffic_Videos/resultant_dataset/yolo11m.pt")  # You can choose a different model size like yolov8s.pt, yolov8m.pt, etc.

model = YOLO("/content/resultant_dataset/yolo11m.pt")  # You can choose a different model size like yolov8s.pt, yolov8m.pt, etc.
# Train the model on the custom dataset
# Assuming your dataset has a data.yaml file in the extracted directory
#base_resultant_dir = "/content/drive/MyDrive/Traffic_Videos/resultant_dataset"

data_config_path = '/content/resultant_dataset/data.yaml'

if os.path.exists(data_config_path):
    results = model.train(data=data_config_path, epochs=20, imgsz=640) # You can adjust epochs and imgsz
    print("Training finished.")
else:
    print(f"Error: data.yaml not found at {data_config_path}. Please ensure your dataset is correctly extracted and contains a data.yaml file.")


Ultralytics 8.3.182 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/resultant_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/resultant_dataset/yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

train: Scanning /content/resultant_dataset/train/labels... 12131 images, 3 backgrounds, 0 corrupt: 100%|██████████| 12131/12131 [00:05<00:00, 2152.07it/s]


train: New cache created: /content/resultant_dataset/train/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 91, len(boxes) = 44377. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 688.2±482.2 MB/s, size: 55.0 KB)


val: Scanning /content/resultant_dataset/valid/labels... 2404 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2404/2404 [00:01<00:00, 1355.89it/s]

val: New cache created: /content/resultant_dataset/valid/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 39, len(boxes) = 7108. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      20/20      8.16G      1.117     0.6281      1.198          6        640: 100%|██████████| 759/759 [07:06<00:00,  1.78it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 76/76 [00:34<00:00,  2.20it/s]


                   all       2404       7108      0.785       0.76      0.802      0.518

20 epochs completed in 2.592 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 40.5MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 40.5MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.182 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11m summary (fused): 125 layers, 20,036,971 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 76/76 [00:35<00:00,  2.12it/s]


                   all       2404       7108      0.785       0.76      0.802      0.518
                   bus        467        702      0.798      0.846      0.894      0.598
                   car        592       2090      0.732      0.733       0.78      0.418
              microbus        295        375      0.647      0.779      0.786      0.496
             motorbike        509       1276      0.743      0.618      0.726      0.302
            pickup-van        271        335      0.707       0.71      0.741      0.452
                 truck         59         70      0.635      0.596      0.613      0.371
           green-light        595        883      0.922      0.741      0.814      0.593
             red-light        584        908      0.892      0.827       0.87      0.605
          yellow-light        434        469      0.987      0.986      0.992       0.83
Speed: 0.2ms preprocess, 10.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detec